# 02 — Auditoria de dados e reprodução dos resultados científicos

Requer o **CSV público original** (`MetroPT3(AirCompressor).csv`, ~209 MB) no Google Drive ou na pasta `dados/`. A execução pode levar alguns minutos no Colab. O script original da Etapa 4 gera as tabelas citadas no artigo: ciclos, alertas, comparação dos critérios e avaliação por evento.

In [ ]:
# Preparação: se executado a partir de um notebook aberto no Colab,
# carregue o ZIP do repositório gerado neste projeto (ou clone-o via GitHub).
from pathlib import Path
import os, sys, zipfile, subprocess
BASE = Path.cwd()
if not (BASE / 'iot_monitor').exists():
    git_repo = ""  # após publicar, opcional: "https://github.com/SEU_USUARIO/railpulse-metropt3.git"
    if git_repo.strip():
        subprocess.run(['git', 'clone', git_repo, '/content/railpulse-metropt3'], check=True)
        BASE = Path('/content/railpulse-metropt3')
    else:
        try:
            from google.colab import files
        except ImportError as e:
            raise RuntimeError('Abra o notebook na raiz do repositório ou no Colab e envie o ZIP do projeto.') from e
        print('Selecione o ZIP do repositório RailPulse. NÃO selecione o CSV original agora.')
        enviados = files.upload()
        nome = next((n for n in enviados if n.endswith('.zip')), None)
        assert nome, 'É necessário carregar o ZIP do projeto.'
        BASE = Path('/content/railpulse-metropt3')
        BASE.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(nome) as z:
            z.extractall(BASE)
os.chdir(BASE)
sys.path.insert(0, str(BASE))
print('Raiz do projeto:', BASE)

In [ ]:
# CSV público original: o arquivo NÃO acompanha o GitHub.
# No Colab, recomendamos colocá-lo no Google Drive para evitar reenvios.
from pathlib import Path
CSV = BASE / 'dados' / 'MetroPT3(AirCompressor).csv'
if not CSV.exists():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        candidato = Path('/content/drive/MyDrive/MetroPT3(AirCompressor).csv')
        if candidato.exists(): CSV = candidato
    except ImportError:
        pass
assert CSV.exists(), 'Coloque MetroPT3(AirCompressor).csv em dados/ ou na raiz do Meu Drive.'
print('CSV:', CSV, 'MB:', round(CSV.stat().st_size/1e6, 1))

## 1. Conferir a proveniência, as lacunas e o conteúdo

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'scripts/auditar_csv.py','--csv',str(CSV)],check=True)

## 2. Executar a análise completa dos ciclos (Etapa 4)

In [ ]:
OUT=BASE/'resultados'/'etapa4'
OUT.mkdir(parents=True,exist_ok=True)
subprocess.run([sys.executable,'analises/etapa4_ciclos.py','--csv',str(CSV),'--out',str(OUT)],check=True)

## 3. Conferir os valores com o artigo

In [ ]:
import pandas as pd
from IPython.display import display
comparacao=pd.read_csv(OUT/'comparacao_regras.csv')
parametros=pd.read_json(OUT/'parametros.json',typ='series')
display(comparacao)
print('Tempo de recuperação P99:',parametros['q99_tempo_ate_9_7_s'])
print('Ciclos:',parametros['n_fases_carga'],'elegíveis:',parametros['n_ciclos_inicio_nominal_validos'])
assert parametros['original_n']==1516948
assert parametros['n_fases_carga']==15769
assert parametros['n_ciclos_inicio_nominal_validos']==10417
for metodo,n in [('Pressao_nao_recuperada_q99',33),('Carga_longa_q99',21),('Somente_carga_mesmo_limiar',136)]:
    assert int(comparacao.loc[comparacao.metodo==metodo,'total_alertas_teste'].iloc[0])==n
print('✅ Os 190 alertas das três regras foram reproduzidos.')

## 4. Verificação de causalidade temporal

In [ ]:
subprocess.run([sys.executable,'analises/verificar_etapa4.py','--csv',str(CSV),'--out',str(OUT)],check=True)